### Importing the libraries

In [129]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, max_error
from tabulate import tabulate
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import MinMaxScaler, RobustScaler, MaxAbsScaler, StandardScaler, FunctionTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import PowerTransformer
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from scipy.stats import boxcox
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from scikeras.wrappers import KerasRegressor


### Load data from CSV file

In [130]:
restaurants_df = pd.read_csv("../Foursquare/final_restaurants_dataset_cleaned.csv")
coffeeshops_df = pd.read_csv("../Foursquare/final_coffeeshops_dataset_cleaned.csv")
print(restaurants_df.info())
print(coffeeshops_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1221 entries, 0 to 1220
Data columns (total 39 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Business ID                               1221 non-null   object 
 1   Name                                      1221 non-null   object 
 2   Latitude                                  1221 non-null   float64
 3   Longitude                                 1221 non-null   float64
 4   Category                                  1221 non-null   object 
 5   Rating                                    1221 non-null   object 
 6   Popularity                                1221 non-null   float64
 7   Google Place ID                           1221 non-null   object 
 8   Business Status                           1221 non-null   object 
 9   Distance (m)                              1221 non-null   float64
 10  Cluster                             

### Combine and convert generalCategory to numerical

In [131]:
combined_df = pd.concat([coffeeshops_df, restaurants_df], ignore_index=True)
combined_df["generalCategory"] = combined_df["generalCategory"].map({"coffee shop": 0, "restaurant": 1})

# Pre-processing

In [132]:
# Function to remove highly correlated features
def remove_highly_correlated_features(X, threshold=0.9):
    """Removes highly correlated features from X."""
    corr_matrix = X.corr().abs()  # Compute absolute correlation matrix
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))  # Upper triangle

    # Find columns to drop
    to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
    return X.drop(columns=to_drop, errors='ignore'), to_drop  # Drop and return removed features

results = []

### Filling missing values with the median

In [133]:
combined_df["Avg Rating - Business Type"] = combined_df["Avg Rating - Business Type"].fillna(combined_df["Avg Rating - Business Type"].median())
combined_df["Avg Rating - Food & Dining"] = combined_df["Avg Rating - Food & Dining"].fillna(combined_df["Avg Rating - Food & Dining"].median())

### Drop irrelevent columns

In [134]:
irrelevant_columns = ['Business ID', 'Latitude'	, 'Longitude', 'Name', 'Category', 'Rating', 'Google Place ID', 'Business Status',
                'Cluster', 'Distance (m)', 'Avg Rating - Business Type', 'Competition - Business Type/Area', 'Competition - Food & Dining/Area',
                'Competition - Business Type/POI Density', 'Competition - Food & Dining/POI Density', 'Competition - Business Type/related POIs'
               , 'Competition - Food & Dining/POI Density','Business Name' ,'Google Rating', 'Popularity']
combined_df.drop(columns=irrelevant_columns, inplace=True)

# SVR

In [135]:
# Ensure that the 'Number of Reviewers' column is numeric and handle errors as NaN
combined_df['Number of Reviewers'] = pd.to_numeric(combined_df['Number of Reviewers'], errors='coerce')

# Drop rows where 'Number of Reviewers' is NaN (if you want to ignore them in analysis)
combined_df = combined_df.dropna(subset=['Number of Reviewers'])


In [136]:
transformed_df = combined_df.copy()

# Remove outliers in 'Number of Reviewers' using IQR method
Q1 = transformed_df['Number of Reviewers'].quantile(0.25)
Q3 = transformed_df['Number of Reviewers'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
transformed_df = transformed_df[(transformed_df['Number of Reviewers'] >= lower_bound) & (transformed_df['Number of Reviewers'] <= upper_bound)]

# Remove outliers in 'Population Within 1km' using IQR method
Q1 = transformed_df['Population Within 1km'].quantile(0.25)
Q3 = transformed_df['Population Within 1km'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
transformed_df = transformed_df[(transformed_df['Population Within 1km'] >= lower_bound) & (transformed_df['Population Within 1km'] <= upper_bound)]

# Remove outliers in 'Restaurants' using IQR method
Q1 = transformed_df['Restaurants'].quantile(0.25)
Q3 = transformed_df['Restaurants'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
transformed_df = transformed_df[(transformed_df['Restaurants'] >= lower_bound) & (transformed_df['Restaurants'] <= upper_bound)]


num_cols = transformed_df.select_dtypes(include=['number']).columns
# Apply PowerTransformer (normalize distribution)
power_transformer = PowerTransformer()
transformed_df[num_cols] = power_transformer.fit_transform(transformed_df[num_cols])

In [137]:
# Apply MaxAbsScaler 
minmax_scaler = MaxAbsScaler()
transformed_df[num_cols] = minmax_scaler.fit_transform(transformed_df[num_cols])

In [126]:
# num_cols = transformed_df.select_dtypes(include=['number']).columns

# def transform_skewed_features(df, skew_threshold=1):
#     transformed_df = df.copy()
    
#     for col in df.select_dtypes(include=[np.number]).columns:  # Only apply to numerical columns
#         skewness = df[col].skew()

#         if skewness < -skew_threshold:  # Highly left-skewed
#             print(f"Applying square transformation to {col}")
#             transformed_df[col] = df[col] ** 2
        
#         elif skewness > skew_threshold:  # Highly right-skewed
#             if (df[col] > 0).all():  # Box-Cox requires positive values
#                 print(f"Applying Box-Cox transformation to {col}")
#                 transformed_df[col], _ = boxcox(df[col])  # Box-Cox transformation
#             else:
#                 print(f"Applying log transformation to {col}")
#                 transformed_df[col] = np.log1p(df[col])  # Log transformation
            
#         else:
#             print(f"No transformation applied to {col} (skew: {skewness:.3f})")
    
#     return transformed_df

# transformed_df = transform_skewed_features(transformed_df)

No transformation applied to generalCategory (skew: -0.449)
Applying log transformation to Religious Institutions
Applying log transformation to Coffee Shops
No transformation applied to Food & Dining (skew: 0.321)
No transformation applied to Restaurants (skew: 0.388)
Applying log transformation to Home & Construction Services
Applying log transformation to Entertainment & Recreation
Applying Box-Cox transformation to Retail & Shopping
No transformation applied to Finance & Services (skew: 0.594)
Applying log transformation to Education
Applying log transformation to Health
Applying log transformation to Public & Government Services
Applying log transformation to Hotels & Hospitality
Applying log transformation to Transportation & Travel
Applying log transformation to Beauty & Wellness
Applying square transformation to POI Density
No transformation applied to Avg Rating - Food & Dining (skew: -0.404)
No transformation applied to Competition - Food & Dining/related POIs (skew: -0.051)


PowerTransformer + MaxAbs 

In [138]:
# Assume 'number_of_reviewers' is the target column
X = transformed_df.drop(columns=['Number of Reviewers'])
y = transformed_df['Number of Reviewers']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Function to create model
def create_model(neurons=64, dropout_rate=0.2, optimizer='adam'):
    model = Sequential([
        Input(shape=(X_train.shape[1],)),  # Explicit Input layer
        Dense(neurons, activation='relu'),
        Dropout(dropout_rate),
        Dense(neurons // 2, activation='relu'),
        Dropout(dropout_rate),
        Dense(1)  # Regression output
    ])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

# Wrap Keras model with KerasRegressor
model = KerasRegressor(model=create_model, verbose=0)

# Define hyperparameters grid
param_grid = {
    'model__neurons': [32, 64, 128],
    'model__dropout_rate': [0.1, 0.2, 0.3],
    'model__optimizer': ['adam', 'rmsprop'],
    'batch_size': [16, 32],
    'epochs': [50, 100]
}

# Perform GridSearchCV
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='neg_mean_absolute_error', cv=3, verbose=2)
grid_result = grid.fit(X_train, y_train)

print(grid.cv_results_['mean_test_score'])

# Best hyperparameters
print(f"Best parameters: {grid_result.best_params_}")

# Train the best model
best_model = grid_result.best_estimator_
best_model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=1)

# Evaluate the model
y_pred = best_model.predict(X_test)

# Calculate and print evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)
medae = median_absolute_error(y_test, y_pred)
max_err = max_error(y_test, y_pred)
mad = np.mean(np.abs(y_test - np.mean(y_test)))

# Print results in table format
print(f"{'Metric':<15} │ {'Value':>10}")
print("-" * 35)
print(f"{'RMSE':<15} │ {rmse:>10.4f}")
print(f"{'MSE':<15} │ {mse:>10.4f}")
print(f"{'MAE':<15} │ {mae:>10.4f}")
print(f"{'R²':<15} │ {r2:>10.4f}")
print(f"{'MedAE':<15} │ {medae:>10.4f}")
print(f"{'Max Error':<15} │ {max_err:>10.4f}")
print(f"{'MAD':<15} │ {mad:>10.4f}")

print("Model evaluation completed!")

Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=adam; total time=   8.8s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=adam; total time=   8.4s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=adam; total time=   8.2s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=rmsprop; total time=   7.6s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=rmsprop; total time=   7.7s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=rmsprop; total time=   7.5s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=64, model__optimizer=adam; total time=   7.9s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=64, mod

In [127]:
# Apply MinMaxScaler (scale to 0-1)
minmax_scaler = MaxAbsScaler()
transformed_df[num_cols] = minmax_scaler.fit_transform(transformed_df[num_cols])

With different transformation methods + MaxAbsScaler 

In [128]:
# Assume 'number_of_reviewers' is the target column
X = transformed_df.drop(columns=['Number of Reviewers'])
y = transformed_df['Number of Reviewers']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Function to create model
def create_model(neurons=64, dropout_rate=0.2, optimizer='adam'):
    model = Sequential([
        Input(shape=(X_train.shape[1],)),  # Explicit Input layer
        Dense(neurons, activation='relu'),
        Dropout(dropout_rate),
        Dense(neurons // 2, activation='relu'),
        Dropout(dropout_rate),
        Dense(1)  # Regression output
    ])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

# Wrap Keras model with KerasRegressor
model = KerasRegressor(model=create_model, verbose=0)

# Define hyperparameters grid
param_grid = {
    'model__neurons': [32, 64, 128],
    'model__dropout_rate': [0.1, 0.2, 0.3],
    'model__optimizer': ['adam', 'rmsprop'],
    'batch_size': [16, 32],
    'epochs': [50, 100]
}

# Perform GridSearchCV
grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring='neg_mean_absolute_error', cv=3, verbose=2)
grid_result = grid.fit(X_train, y_train)

print(grid.cv_results_['mean_test_score'])

# Best hyperparameters
print(f"Best parameters: {grid_result.best_params_}")

# Train the best model
best_model = grid_result.best_estimator_
best_model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=1)

# Evaluate the model
y_pred = best_model.predict(X_test)

# Calculate and print evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)
medae = median_absolute_error(y_test, y_pred)
max_err = max_error(y_test, y_pred)
mad = np.mean(np.abs(y_test - np.mean(y_test)))

# Print results in table format
print(f"{'Metric':<15} │ {'Value':>10}")
print("-" * 35)
print(f"{'RMSE':<15} │ {rmse:>10.4f}")
print(f"{'MSE':<15} │ {mse:>10.4f}")
print(f"{'MAE':<15} │ {mae:>10.4f}")
print(f"{'R²':<15} │ {r2:>10.4f}")
print(f"{'MedAE':<15} │ {medae:>10.4f}")
print(f"{'Max Error':<15} │ {max_err:>10.4f}")
print(f"{'MAD':<15} │ {mad:>10.4f}")

print("Model evaluation completed!")



Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=adam; total time=  10.3s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=adam; total time=   9.5s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=adam; total time=  10.0s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=rmsprop; total time=   9.4s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=rmsprop; total time=   9.2s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=32, model__optimizer=rmsprop; total time=   9.5s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=64, model__optimizer=adam; total time=   9.4s
[CV] END batch_size=16, epochs=50, model__dropout_rate=0.1, model__neurons=64, mod